# Lecture 08: High-Dimensional Stochastic Optimal Control

**Course:** Computational Mathematics and AI

This notebook demonstrates neural network methods for solving high-dimensional stochastic optimal control problems, specifically the 100D benchmark from the Deep BSDE literature.

## Key Insight

> **"The sampling strategy, not just the neural network, determines success!"**

We compare three approaches:
1. **PINN** - Physics-Informed Neural Networks (HJB collocation)
2. **FBSNN** - Forward-Backward Stochastic Neural Networks (BSDE formulation, random walk)
3. **NeuralSOC** - Neural Stochastic Optimal Control (PMP-guided sampling)

The key finding: FBSNN fails on shifted targets because random walk sampling doesn't explore the relevant state space!

In [1]:
# Colab setup - run this cell first if using Google Colab
import sys, os

if 'google.colab' in sys.modules:
    repo_dir = '/content/CompMathAndAICourse/'
    if not os.path.exists(repo_dir):
        !git clone https://github.com/lruthotto/CompMathAndAICourse.git {repo_dir}
        %pip install -q jax jaxlib equinox diffrax optax pyyaml

    sys.path.insert(0, repo_dir + 'workspace/code/08-stochastic-control')
    os.chdir(repo_dir + 'workspace/code/08-stochastic-control')
    print(f"Running on Colab, repo cloned to {repo_dir}")

In [2]:
# =============================================================================
# Configuration
# =============================================================================
# Set TRAIN_MODE = False to skip training and load saved models for plot regeneration
TRAIN_MODE = True

# Set INTERACTIVE = True for Jupyter (shows figures inline), False for headless/saving only
INTERACTIVE = False

# =============================================================================
# Imports and Setup
# =============================================================================
import matplotlib
if not INTERACTIVE:
    matplotlib.use('Agg')

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pickle

# Local modules
from problem import (
    ProblemConfig, create_problem_config, DEFAULT_CONFIG, SHIFTED_CONFIG,
    terminal_cost, analytical_solution_mc, sample_initial_states,
    evaluate_control_objective,
)
from networks import create_value_network, PhiResNet
from solvers import create_solver, PINNSolver, FBSNNSolver, NeuralSOCSolver
from trainer import train_solver, TrainingConfig, Trainer
from visualization import (
    plot_training_convergence, plot_value_function_slice,
    plot_value_function_diagonal_slice, compute_value_function_2d_diagonal_slice,
    plot_trajectories, plot_control_components, plot_method_comparison,
    save_results_csv, save_training_convergence_csv, save_convergence_comparison_csv,
    create_full_results_figure, COLORS,
)
from config_loader import (
    load_method_config, load_all_configs, print_config, MethodConfig
)

# Check JAX backend
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")
print(f"Mode: {'TRAINING' if TRAIN_MODE else 'PLOT REGENERATION (loading saved models)'}")
print(f"Figures: {'INTERACTIVE' if INTERACTIVE else 'SAVING TO FILES'}")

# Master random key for reproducibility
MASTER_KEY = jr.PRNGKey(42)

# Create output directories
NOTEBOOK_DIR = Path.cwd()
OUTPUT_DIR = NOTEBOOK_DIR / "figures"
MODEL_DIR = NOTEBOOK_DIR / "saved_models"
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Model directory: {MODEL_DIR}")


# =============================================================================
# Helper functions for saving/loading models
# =============================================================================
def save_model(params, filepath):
    """Save JAX model parameters to file using pickle."""
    with open(filepath, 'wb') as f:
        pickle.dump(params, f)
    print(f"Saved model to: {filepath}")


def load_model(filepath):
    """Load JAX model parameters from file."""
    with open(filepath, 'rb') as f:
        params = pickle.load(f)
    print(f"Loaded model from: {filepath}")
    return params


def show_or_save(fig, filepath, title=None):
    """Show figure interactively or save to file based on INTERACTIVE mode."""
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    if INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)
    print(f"Saved: {filepath}")

JAX version: 0.8.1
JAX devices: [CpuDevice(id=0)]
Mode: TRAINING
Figures: SAVING TO FILES
Output directory: /workspace/code/08-stochastic-control/figures
Model directory: /workspace/code/08-stochastic-control/saved_models


## 1. Problem Setup

### The 100D Benchmark Problem

**Stochastic Optimal Control:**
$$\min_u J = \mathbb{E}\left[\int_0^T \|u\|^2 \, dt + g(X_T)\right]$$
$$\text{subject to:} \quad dX = 2u \, dt + \sqrt{2} \, dW$$

**Terminal cost options:**
- Log cost: $g(x) = \log\left(\frac{1 + \|x - x_{\text{target}}\|^2}{2}\right)$
- Quadratic cost: $g(x) = \|x - x_{\text{target}}\|^2$

**HJB Equation:**
$$-\frac{\partial \Phi}{\partial t} + \Delta \Phi - \|\nabla \Phi\|^2 = 0, \quad \Phi(T,x) = g(x)$$

**Optimal Control (from Pontryagin Maximum Principle):**
$$u^* = -\nabla_x \Phi$$

In [3]:
# =============================================
# Configuration - MODIFY THESE PARAMETERS
# =============================================

# Problem settings
D = 100                    # State dimension
T = 1.0                    # Terminal time
SHIFTED_TARGET = False     # If True, target at (3,3,...,3); else at origin

# Use tuned hyperparameters from HPO?
USE_TUNED_CONFIGS = True   # If True, load from config/*.yaml files

# Override max iterations (useful for quick tests)
MAX_ITERATIONS_OVERRIDE = 1000  # Set to e.g. 1000 for quick runs, None to use tuned value

# =============================================
# Load configurations
# =============================================
if USE_TUNED_CONFIGS:
    print("Loading tuned configurations from YAML files...")
    configs = load_all_configs(shifted=SHIFTED_TARGET, config_dir='config')
    pinn_cfg = configs['pinn']
    fbsnn_cfg = configs['fbsnn']
    neural_soc_cfg = configs['neural_soc']

    # Print loaded configs
    for method, cfg in configs.items():
        print_config(cfg, method)
else:
    # Use default configurations
    print("Using default configurations...")
    from config_loader import DEFAULT_CONFIGS
    pinn_cfg = DEFAULT_CONFIGS['pinn']
    fbsnn_cfg = DEFAULT_CONFIGS['fbsnn']
    neural_soc_cfg = DEFAULT_CONFIGS['neural_soc']

# Apply iteration override if specified
if MAX_ITERATIONS_OVERRIDE is not None:
    print(f"\nOverriding max_iterations to {MAX_ITERATIONS_OVERRIDE}")
    pinn_cfg = MethodConfig(**{**pinn_cfg.__dict__, 'max_iterations': MAX_ITERATIONS_OVERRIDE})
    fbsnn_cfg = MethodConfig(**{**fbsnn_cfg.__dict__, 'max_iterations': MAX_ITERATIONS_OVERRIDE})
    neural_soc_cfg = MethodConfig(**{**neural_soc_cfg.__dict__, 'max_iterations': MAX_ITERATIONS_OVERRIDE})

# Terminal cost type depends on problem
TERMINAL_COST_TYPE = "quadratic" if SHIFTED_TARGET else "log"

# Create problem configuration
config = create_problem_config(
    d=D, T=T,
    shifted_target=SHIFTED_TARGET,
    terminal_cost_type=TERMINAL_COST_TYPE,
)

# Model paths for saving/loading
PINN_MODEL_PATH = MODEL_DIR / f'pinn_params{"_shifted" if SHIFTED_TARGET else ""}.pkl'
FBSNN_MODEL_PATH = MODEL_DIR / f'fbsnn_params{"_shifted" if SHIFTED_TARGET else ""}.pkl'
NEURAL_SOC_MODEL_PATH = MODEL_DIR / f'neural_soc_params{"_shifted" if SHIFTED_TARGET else ""}.pkl'

print("\n" + "="*60)
print("Problem Configuration:")
print("="*60)
print(f"  Dimension d = {config.d}")
print(f"  Terminal time T = {config.T}")
print(f"  Diffusion sigma = {config.sigma:.4f}")
print(f"  Terminal cost type: {config.terminal_cost_type}")
if config.x_target is not None:
    print(f"  Target: x_target[0] = {config.x_target[0]:.1f} (shifted)")
else:
    print(f"  Target: origin (default)")
print(f"\nModel save/load paths:")
print(f"  PINN: {PINN_MODEL_PATH}")
print(f"  FBSNN: {FBSNN_MODEL_PATH}")
print(f"  NeuralSOC: {NEURAL_SOC_MODEL_PATH}")

Loading tuned configurations from YAML files...

PINN Configuration:
  Network: resnet, width=256, depth=3
  Solver: n_steps=50
  Training: lr=0.021830, batch=32
  LR schedule: decay=0.0768, steps=1500
  Loss weights: (1.0, 1.0)
  HPO best value: 0.0037 (tuned 2025-12-04)

FBSNN Configuration:
  Network: resnet, width=64, depth=2
  Solver: n_steps=30
  Training: lr=0.006741, batch=256
  LR schedule: decay=0.3156, steps=2000
  Loss weights: (0.2987274199563841, 2.860439042982462, 0.3677831327192532)
  HPO best value: 4.5806 (tuned 2025-12-10)

NEURAL_SOC Configuration:
  Network: resnet, width=256, depth=6
  Solver: n_steps=10
  Training: lr=0.000906, batch=256
  LR schedule: decay=0.9230, steps=1000
  Loss weights: (1.0, 1.0)

Overriding max_iterations to 1000

Problem Configuration:
  Dimension d = 100
  Terminal time T = 1.0
  Diffusion sigma = 1.4142
  Terminal cost type: log
  Target: origin (default)

Model save/load paths:
  PINN: /workspace/code/08-stochastic-control/saved_model

In [4]:
# Compute analytical optimal value at x0=0
key = MASTER_KEY
key, opt_key = jr.split(key)
x0 = jnp.zeros(config.d)

optimal_value = float(analytical_solution_mc(0.0, x0, config, opt_key, n_samples=50000))
print(f"\nAnalytical optimal value Phi(0, 0) = {optimal_value:.6f}")

# Terminal cost at origin and target
g_at_origin = float(terminal_cost(jnp.zeros(config.d), config))
if config.x_target is not None:
    g_at_target = float(terminal_cost(config.x_target, config))
    print(f"Terminal cost g(origin) = {g_at_origin:.6f}")
    print(f"Terminal cost g(target) = {g_at_target:.6f}")
else:
    print(f"Terminal cost g(origin) = {g_at_origin:.6f}")


Analytical optimal value Phi(0, 0) = 4.590336
Terminal cost g(origin) = -0.693147


## 2. Method 1: PINN (Physics-Informed Neural Network)

**Loss function:** HJB residual + terminal condition
$$\mathcal{L}_{\text{PINN}} = \mathbb{E}\left[\left|-\frac{\partial \Phi}{\partial t} + \Delta\Phi - \|\nabla\Phi\|^2\right|^2\right] + \lambda \|\Phi(T,\cdot) - g\|^2$$

**Sampling:** Random walk (trajectory-based) or uniform

In [5]:
# Create PINN solver (using tuned config)
key, net_key = jr.split(key)
phi_net_pinn = create_value_network(
    net_key, D, architecture=pinn_cfg.architecture,
    hidden_width=pinn_cfg.hidden_width, depth=pinn_cfg.depth
)

pinn_solver = create_solver(
    'pinn', phi_net_pinn, config, n_steps=pinn_cfg.n_steps,
    sampling_mode='trajectory',  # Use trajectory-based sampling
    use_trace_estimator=True,    # Hutchinson trace estimator (O(d), memory efficient)
)

print("PINN Solver created")
print(f"  Network params: {sum(p.size for p in jax.tree.leaves(phi_net_pinn) if hasattr(p, 'size')):,}")
print(f"  Architecture: {pinn_cfg.architecture}, width={pinn_cfg.hidden_width}, depth={pinn_cfg.depth}")

PINN Solver created
  Network params: 223,745
  Architecture: resnet, width=256, depth=3


In [6]:
# Train PINN (or load saved model)
if TRAIN_MODE:
    print("\nTraining PINN...")
    key, train_key = jr.split(key)

    trained_pinn, history_pinn = train_solver(
        pinn_solver, config, method='pinn', key=train_key,
        max_iterations=pinn_cfg.max_iterations,
        batch_size=pinn_cfg.batch_size,
        learning_rate=pinn_cfg.learning_rate,
        lr_decay=pinn_cfg.lr_decay,
        lr_decay_steps=pinn_cfg.lr_decay_steps,
        loss_weights=pinn_cfg.loss_weights,
        print_freq=pinn_cfg.print_freq,
        val_freq=pinn_cfg.val_freq,
        patience=pinn_cfg.patience,
        verbose=True
    )
    
    # Save trained model
    save_model(trained_pinn, PINN_MODEL_PATH)
    
    # Save training history
    import pandas as pd
    pinn_history_df = pd.DataFrame({
        'iteration': history_pinn.iterations,
        'loss': history_pinn.losses,
    })
    pinn_history_df.to_csv(OUTPUT_DIR / 'pinn_training_history.csv', index=False)
    print(f"Saved: {OUTPUT_DIR / 'pinn_training_history.csv'}")
else:
    # Load saved model
    if PINN_MODEL_PATH.exists():
        trained_pinn = load_model(PINN_MODEL_PATH)
        # Load training history if available
        pinn_history_path = OUTPUT_DIR / 'pinn_training_history.csv'
        if pinn_history_path.exists():
            import pandas as pd
            pinn_history_df = pd.read_csv(pinn_history_path)
            # Create a minimal history object for compatibility
            class MinimalHistory:
                def __init__(self, df):
                    self.iterations = df['iteration'].tolist()
                    self.losses = df['loss'].tolist()
                    self.final_loss = self.losses[-1] if self.losses else None
                    self.final_rel_error = None
                    self.final_suboptimality = None
            history_pinn = MinimalHistory(pinn_history_df)
            print(f"Loaded training history from: {pinn_history_path}")
        else:
            history_pinn = None
            print("Warning: No training history found")
    else:
        raise FileNotFoundError(f"No saved model found at {PINN_MODEL_PATH}. Run with TRAIN_MODE=True first.")


Training PINN...


Iter     0 | Loss: 18.746035 | Best: 18.746035 | Time: 3.3s


  Val | Rel Error: 0.7560 | Subopt: 2.6999


Iter   200 | Loss: 0.172262 | Best: 0.157969 | Time: 109.1s


Iter   400 | Loss: 0.059584 | Best: 0.037287 | Time: 204.3s


  Val | Rel Error: 0.0234 | Subopt: 0.0075


Iter   600 | Loss: 0.028852 | Best: 0.025648 | Time: 308.3s


Iter   800 | Loss: 0.050994 | Best: 0.018334 | Time: 409.6s


## 3. Method 2: FBSNN (Forward-Backward Stochastic Neural Network)

**Loss function:** BSDE residual + terminal conditions
$$\mathcal{L}_{\text{FBSNN}} = \mathbb{E}\left[\sum_k |Y_{k+1} - Y_k - f_k \Delta t - Z_k \cdot \Delta W_k|^2\right] + \lambda_1 |\Phi(T,\cdot) - g|^2 + \lambda_2 |Z_T - \sigma \nabla g|^2$$

**Sampling:** Pure random walk (dX = σdW, no drift!)

⚠️ **Warning:** FBSNN FAILS on shifted targets because random walk sampling doesn't reach the target region!

In [7]:
# Create FBSNN solver (using tuned config)
key, net_key = jr.split(key)
phi_net_fbsnn = create_value_network(
    net_key, D, architecture=fbsnn_cfg.architecture,
    hidden_width=fbsnn_cfg.hidden_width, depth=fbsnn_cfg.depth
)

fbsnn_solver = create_solver(
    'fbsnn', phi_net_fbsnn, config, n_steps=fbsnn_cfg.n_steps
)

print("FBSNN Solver created")
print(f"  Network params: {sum(p.size for p in jax.tree.leaves(phi_net_fbsnn) if hasattr(p, 'size')):,}")
print(f"  Architecture: {fbsnn_cfg.architecture}, width={fbsnn_cfg.hidden_width}, depth={fbsnn_cfg.depth}")

FBSNN Solver created
  Network params: 14,913
  Architecture: resnet, width=64, depth=2


In [ ]:
# Train FBSNN (or load saved model)
if TRAIN_MODE:
    print("\nTraining FBSNN...")
    key, train_key = jr.split(key)

    trained_fbsnn, history_fbsnn = train_solver(
        fbsnn_solver, config, method='fbsnn', key=train_key,
        max_iterations=fbsnn_cfg.max_iterations,
        batch_size=fbsnn_cfg.batch_size,
        learning_rate=fbsnn_cfg.learning_rate,
        lr_decay=fbsnn_cfg.lr_decay,
        lr_decay_steps=fbsnn_cfg.lr_decay_steps,
        loss_weights=fbsnn_cfg.loss_weights,
        print_freq=fbsnn_cfg.print_freq,
        val_freq=fbsnn_cfg.val_freq,
        patience=fbsnn_cfg.patience,
        verbose=True
    )
    
    # Save trained model
    save_model(trained_fbsnn, FBSNN_MODEL_PATH)
    
    # Save training history
    import pandas as pd
    fbsnn_history_df = pd.DataFrame({
        'iteration': history_fbsnn.iterations,
        'loss': history_fbsnn.losses,
    })
    fbsnn_history_df.to_csv(OUTPUT_DIR / 'fbsnn_training_history.csv', index=False)
    print(f"Saved: {OUTPUT_DIR / 'fbsnn_training_history.csv'}")
else:
    # Load saved model
    if FBSNN_MODEL_PATH.exists():
        trained_fbsnn = load_model(FBSNN_MODEL_PATH)
        # Load training history if available
        fbsnn_history_path = OUTPUT_DIR / 'fbsnn_training_history.csv'
        if fbsnn_history_path.exists():
            import pandas as pd
            fbsnn_history_df = pd.read_csv(fbsnn_history_path)
            class MinimalHistory:
                def __init__(self, df):
                    self.iterations = df['iteration'].tolist()
                    self.losses = df['loss'].tolist()
                    self.final_loss = self.losses[-1] if self.losses else None
                    self.final_rel_error = None
                    self.final_suboptimality = None
            history_fbsnn = MinimalHistory(fbsnn_history_df)
            print(f"Loaded training history from: {fbsnn_history_path}")
        else:
            history_fbsnn = None
            print("Warning: No training history found")
    else:
        raise FileNotFoundError(f"No saved model found at {FBSNN_MODEL_PATH}. Run with TRAIN_MODE=True first.")

## 4. Method 3: NeuralSOC (Neural Stochastic Optimal Control)

**Loss function:** Control objective + terminal matching
$$\mathcal{L}_{\text{NeuralSOC}} = \mathbb{E}\left[\int_0^T \|u^*\|^2 dt + g(X_T)\right] + \lambda |\Phi(T,X_T) - g(X_T)|^2$$

**Sampling:** PMP-guided (dX = 2u* dt + σdW, trajectories follow optimal control!)

✅ **Works on shifted targets** because guided sampling explores the relevant region!

In [9]:
# Create NeuralSOC solver (using tuned config)
key, net_key = jr.split(key)
phi_net_neural_soc = create_value_network(
    net_key, D, architecture=neural_soc_cfg.architecture,
    hidden_width=neural_soc_cfg.hidden_width, depth=neural_soc_cfg.depth
)

neural_soc_solver = create_solver(
    'neural_soc', phi_net_neural_soc, config, n_steps=neural_soc_cfg.n_steps
)

print("NeuralSOC Solver created")
print(f"  Network params: {sum(p.size for p in jax.tree.leaves(phi_net_neural_soc) if hasattr(p, 'size')):,}")
print(f"  Architecture: {neural_soc_cfg.architecture}, width={neural_soc_cfg.hidden_width}, depth={neural_soc_cfg.depth}")

NeuralSOC Solver created
  Network params: 355,329
  Architecture: resnet, width=256, depth=5


In [ ]:
# Train NeuralSOC (or load saved model)
if TRAIN_MODE:
    print("\nTraining NeuralSOC...")
    key, train_key = jr.split(key)

    trained_neural_soc, history_neural_soc = train_solver(
        neural_soc_solver, config, method='neural_soc', key=train_key,
        max_iterations=neural_soc_cfg.max_iterations,
        batch_size=neural_soc_cfg.batch_size,
        learning_rate=neural_soc_cfg.learning_rate,
        lr_decay=neural_soc_cfg.lr_decay,
        lr_decay_steps=neural_soc_cfg.lr_decay_steps,
        loss_weights=neural_soc_cfg.loss_weights,
        print_freq=neural_soc_cfg.print_freq,
        val_freq=neural_soc_cfg.val_freq,
        patience=neural_soc_cfg.patience,
        verbose=True
    )
    
    # Save trained model
    save_model(trained_neural_soc, NEURAL_SOC_MODEL_PATH)
    
    # Save training history
    import pandas as pd
    neural_soc_history_df = pd.DataFrame({
        'iteration': history_neural_soc.iterations,
        'loss': history_neural_soc.losses,
    })
    neural_soc_history_df.to_csv(OUTPUT_DIR / 'neural_soc_training_history.csv', index=False)
    print(f"Saved: {OUTPUT_DIR / 'neural_soc_training_history.csv'}")
else:
    # Load saved model
    if NEURAL_SOC_MODEL_PATH.exists():
        trained_neural_soc = load_model(NEURAL_SOC_MODEL_PATH)
        # Load training history if available
        neural_soc_history_path = OUTPUT_DIR / 'neural_soc_training_history.csv'
        if neural_soc_history_path.exists():
            import pandas as pd
            neural_soc_history_df = pd.read_csv(neural_soc_history_path)
            class MinimalHistory:
                def __init__(self, df):
                    self.iterations = df['iteration'].tolist()
                    self.losses = df['loss'].tolist()
                    self.final_loss = self.losses[-1] if self.losses else None
                    self.final_rel_error = None
                    self.final_suboptimality = None
            history_neural_soc = MinimalHistory(neural_soc_history_df)
            print(f"Loaded training history from: {neural_soc_history_path}")
        else:
            history_neural_soc = None
            print("Warning: No training history found")
    else:
        raise FileNotFoundError(f"No saved model found at {NEURAL_SOC_MODEL_PATH}. Run with TRAIN_MODE=True first.")

## 5. Evaluation and Comparison

We evaluate each method by:
1. **Control objective** $J = \mathbb{E}[\int \|u\|^2 dt + g(X_T)]$ (should be close to optimal)
2. **Relative error** vs analytical solution
3. **Trajectory quality** (do trajectories reach the target?)

In [ ]:
# Evaluate all methods
print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)

# Sample evaluation points
key, eval_key = jr.split(key)
x0_eval = sample_initial_states(eval_key, 128, config, scale=0.1)

results = {}

# Evaluate PINN
pinn_solver_trained = create_solver('pinn', trained_pinn, config, n_steps=pinn_cfg.n_steps)
def pinn_policy(t, x):
    return pinn_solver_trained.optimal_control(t, x)

key, eval_key = jr.split(key)
pinn_result = evaluate_control_objective(
    pinn_policy, x0_eval, config, eval_key, n_steps=100
)
results['PINN'] = {
    'control_objective': pinn_result.mean_objective,
    'std_objective': pinn_result.std_objective,
    'optimal_value': pinn_result.optimal_value,
    'relative_suboptimality': pinn_result.relative_suboptimality,
    'final_loss': getattr(history_pinn, 'final_loss', None) if history_pinn else None,
    'final_rel_error': getattr(history_pinn, 'final_rel_error', None) if history_pinn else None,
    'final_suboptimality': getattr(history_pinn, 'final_suboptimality', None) if history_pinn else None,
}
print(f"\nPINN:")
print(f"  Control objective J = {pinn_result.mean_objective:.4f} ± {pinn_result.std_objective:.4f}")
print(f"  Optimal value J* = {pinn_result.optimal_value:.4f}")
print(f"  Relative suboptimality = {pinn_result.relative_suboptimality:.2%}")

# Evaluate FBSNN
fbsnn_solver_trained = create_solver('fbsnn', trained_fbsnn, config, n_steps=fbsnn_cfg.n_steps)
def fbsnn_policy(t, x):
    return fbsnn_solver_trained.optimal_control(t, x)

key, eval_key = jr.split(key)
fbsnn_result = evaluate_control_objective(
    fbsnn_policy, x0_eval, config, eval_key, n_steps=100
)
results['FBSNN'] = {
    'control_objective': fbsnn_result.mean_objective,
    'std_objective': fbsnn_result.std_objective,
    'optimal_value': fbsnn_result.optimal_value,
    'relative_suboptimality': fbsnn_result.relative_suboptimality,
    'final_loss': getattr(history_fbsnn, 'final_loss', None) if history_fbsnn else None,
    'final_rel_error': getattr(history_fbsnn, 'final_rel_error', None) if history_fbsnn else None,
    'final_suboptimality': getattr(history_fbsnn, 'final_suboptimality', None) if history_fbsnn else None,
}
print(f"\nFBSNN:")
print(f"  Control objective J = {fbsnn_result.mean_objective:.4f} ± {fbsnn_result.std_objective:.4f}")
print(f"  Optimal value J* = {fbsnn_result.optimal_value:.4f}")
print(f"  Relative suboptimality = {fbsnn_result.relative_suboptimality:.2%}")

# Evaluate NeuralSOC
neural_soc_solver_trained = create_solver('neural_soc', trained_neural_soc, config, n_steps=neural_soc_cfg.n_steps)
def neural_soc_policy(t, x):
    return neural_soc_solver_trained.optimal_control(t, x)

key, eval_key = jr.split(key)
neural_soc_result = evaluate_control_objective(
    neural_soc_policy, x0_eval, config, eval_key, n_steps=100
)
results['NeuralSOC'] = {
    'control_objective': neural_soc_result.mean_objective,
    'std_objective': neural_soc_result.std_objective,
    'optimal_value': neural_soc_result.optimal_value,
    'relative_suboptimality': neural_soc_result.relative_suboptimality,
    'final_loss': getattr(history_neural_soc, 'final_loss', None) if history_neural_soc else None,
    'final_rel_error': getattr(history_neural_soc, 'final_rel_error', None) if history_neural_soc else None,
    'final_suboptimality': getattr(history_neural_soc, 'final_suboptimality', None) if history_neural_soc else None,
}
print(f"\nNeuralSOC:")
print(f"  Control objective J = {neural_soc_result.mean_objective:.4f} ± {neural_soc_result.std_objective:.4f}")
print(f"  Optimal value J* = {neural_soc_result.optimal_value:.4f}")
print(f"  Relative suboptimality = {neural_soc_result.relative_suboptimality:.2%}")

print(f"\n" + "="*60)
print(f"Optimal value (analytical): {optimal_value:.4f}")

## 6. Visualizations

In [ ]:
# Training convergence comparison
if history_pinn is not None and history_fbsnn is not None and history_neural_soc is not None:
    fig, ax = plt.subplots(figsize=(10, 6))

    ax.semilogy(history_pinn.iterations, history_pinn.losses,
                color=COLORS['pinn'], linewidth=1.5, label='PINN')
    ax.semilogy(history_fbsnn.iterations, history_fbsnn.losses,
                color=COLORS['fbsnn'], linewidth=1.5, label='FBSNN')
    ax.semilogy(history_neural_soc.iterations, history_neural_soc.losses,
                color=COLORS['neural_soc'], linewidth=1.5, label='NeuralSOC')

    ax.set_xlabel('Iteration')
    ax.set_ylabel('Training Loss')
    ax.set_title('Training Convergence Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    show_or_save(fig, OUTPUT_DIR / 'training_convergence.png')
else:
    print("Skipping training convergence plot (history not available)")

In [ ]:
# Method comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

methods = ['PINN', 'FBSNN', 'NeuralSOC']
colors = [COLORS['pinn'], COLORS['fbsnn'], COLORS['neural_soc']]

# Control objective
objectives = [results[m]['control_objective'] for m in methods]
bars1 = axes[0].bar(methods, objectives, color=colors, edgecolor='white', linewidth=1.5)
axes[0].axhline(y=optimal_value, color='red', linestyle='--', linewidth=2, label=f'Optimal J* = {optimal_value:.4f}')
axes[0].set_ylabel('Control Objective J')
axes[0].set_title('Control Objective')
axes[0].legend()

for bar, val in zip(bars1, objectives):
    axes[0].annotate(f'{val:.3f}', xy=(bar.get_x() + bar.get_width()/2, val),
                     xytext=(0, 3), textcoords='offset points', ha='center', fontsize=10)

# Relative suboptimality
subopt = [results[m]['relative_suboptimality'] * 100 for m in methods]
bars2 = axes[1].bar(methods, subopt, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_ylabel('Relative Suboptimality (%)')
axes[1].set_title('Suboptimality vs Optimal')

for bar, val in zip(bars2, subopt):
    axes[1].annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width()/2, val),
                     xytext=(0, 3), textcoords='offset points', ha='center', fontsize=10)

plt.tight_layout()
show_or_save(fig, OUTPUT_DIR / 'method_comparison.png')

In [ ]:
# Trajectory comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

key, k1, k2, k3 = jr.split(key, 4)

plot_trajectories(config, k1, solver=pinn_solver_trained, method='learned',
                  n_trajectories=20, title='PINN Trajectories', ax=axes[0])
plot_trajectories(config, k2, solver=fbsnn_solver_trained, method='learned',
                  n_trajectories=20, title='FBSNN Trajectories', ax=axes[1])
plot_trajectories(config, k3, solver=neural_soc_solver_trained, method='learned',
                  n_trajectories=20, title='NeuralSOC Trajectories', ax=axes[2])

plt.tight_layout()
show_or_save(fig, OUTPUT_DIR / 'trajectory_comparison.png')

In [ ]:
# Value function at t=0
fig, ax = plt.subplots(figsize=(8, 6))

key, plot_key = jr.split(key)
plot_value_function_slice(config, plot_key, t=0.0, title='Analytical Value Function $\\Phi(0, x)$', ax=ax)

plt.tight_layout()
show_or_save(fig, OUTPUT_DIR / 'value_function_t0.png')

In [ ]:
# Value function on diagonal slice (for shifted target problem)
# This 2D plane contains the diagonal from origin (0,...,0) to target (3,...,3)
# Parameterization: x = [x1, x2, (x1+x2)/2, ..., (x1+x2)/2]

if SHIFTED_TARGET:
    # Compute value functions on diagonal slice with consistent colorbar
    key, k1, k2 = jr.split(key, 3)

    # First compute analytical to get colorbar range
    x1, x2, phi_analytical, _ = compute_value_function_2d_diagonal_slice(
        t=0.0, config=config, key=k1, x_range=(-1, 4), n_points=50, n_mc_samples=5000
    )
    vmin, vmax = float(phi_analytical.min()), float(phi_analytical.max())
    print(f"Value function range: [{vmin:.2f}, {vmax:.2f}]")

    # Create figure directory for shifted target
    shifted_dir = OUTPUT_DIR / 'shifted'
    shifted_dir.mkdir(parents=True, exist_ok=True)

    # Plot analytical value function
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_value_function_diagonal_slice(
        config, k1, t=0.0, solver=None, x_range=(-1, 4), n_points=50,
        mode='analytical', vmin=vmin, vmax=vmax,
        title='Analytical Value Function $\\Phi(0, x)$ on Diagonal Slice',
        ax=ax, save_path=None,
        show_diagonal=True
    )
    show_or_save(fig, shifted_dir / 'value_function_diagonal_analytical_t0.png')

    # Plot learned (NeuralSOC) value function
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_value_function_diagonal_slice(
        config, k2, t=0.0, solver=neural_soc_solver_trained, x_range=(-1, 4), n_points=50,
        mode='learned', vmin=vmin, vmax=vmax,
        title='Learned Value Function $\\Phi_{\\theta}(0, x)$ on Diagonal Slice (NeuralSOC)',
        ax=ax, save_path=None,
        show_diagonal=True
    )
    show_or_save(fig, shifted_dir / 'value_function_diagonal_learned_t0.png')
else:
    print("Skipping diagonal slice plots (only for shifted target problem)")

In [ ]:
# Control trajectories for best method
fig, ax = plt.subplots(figsize=(10, 5))

key, ctrl_key = jr.split(key)
plot_control_components(config, neural_soc_solver_trained, ctrl_key,
                        n_trajectories=20, title='NeuralSOC Control Trajectories', ax=ax)

plt.tight_layout()
show_or_save(fig, OUTPUT_DIR / 'control_trajectories.png')

## 7. Save Results

In [ ]:
# Save results to CSV
import pandas as pd

# Final evaluation summary
results_df = pd.DataFrame({
    'Method': ['PINN', 'FBSNN', 'NeuralSOC'],
    'Control_Objective': [results[m]['control_objective'] for m in ['PINN', 'FBSNN', 'NeuralSOC']],
    'Std_Objective': [results[m]['std_objective'] for m in ['PINN', 'FBSNN', 'NeuralSOC']],
    'Optimal_Value': [results[m]['optimal_value'] for m in ['PINN', 'FBSNN', 'NeuralSOC']],
    'Relative_Suboptimality': [results[m]['relative_suboptimality'] for m in ['PINN', 'FBSNN', 'NeuralSOC']],
    'Final_Loss': [results[m]['final_loss'] for m in ['PINN', 'FBSNN', 'NeuralSOC']],
    'Final_Rel_Error': [results[m]['final_rel_error'] for m in ['PINN', 'FBSNN', 'NeuralSOC']],
})

results_df.to_csv(OUTPUT_DIR / 'results_summary.csv', index=False)
print(f"Results saved to {OUTPUT_DIR / 'results_summary.csv'}")
print("\n" + results_df.to_string(index=False))

# Save training convergence CSVs for each method (only if in training mode and histories exist)
print("\n" + "="*60)
if TRAIN_MODE:
    print("Saving training convergence data...")
    save_training_convergence_csv(history_pinn, 'pinn', output_dir=str(OUTPUT_DIR))
    save_training_convergence_csv(history_fbsnn, 'fbsnn', output_dir=str(OUTPUT_DIR))
    save_training_convergence_csv(history_neural_soc, 'neural_soc', output_dir=str(OUTPUT_DIR))

    # Save combined convergence comparison
    histories = {
        'PINN': history_pinn,
        'FBSNN': history_fbsnn,
        'NeuralSOC': history_neural_soc,
    }
    save_convergence_comparison_csv(histories, output_dir=str(OUTPUT_DIR))
    print("\nAll CSV files saved successfully!")
else:
    print("Skipping detailed training history saves (TRAIN_MODE=False)")

# Final summary of saved files
print("\n" + "="*60)
print("SAVED FILES")
print("="*60)
print("\n--- Figures ---")
for f in sorted(OUTPUT_DIR.glob('*.png')):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45} ({size_kb:.1f} KB)")
print("\n--- Models ---")
for f in sorted(MODEL_DIR.glob('*.pkl')):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45} ({size_kb:.1f} KB)")
print("="*60)
print(f"\nTo regenerate plots without retraining:")
print(f"  1. Set TRAIN_MODE = False in the configuration cell")
print(f"  2. Run all cells")

## 8. Summary and Key Takeaways

### Key Findings

1. **Sampling Strategy Matters!**
   - Random walk sampling (FBSNN) fails when the target is shifted
   - PMP-guided sampling (NeuralSOC) works because trajectories explore the relevant region

2. **Method Comparison**
   - **PINN**: Works with trajectory-based sampling, but HJB residual can be expensive (Laplacian)
   - **FBSNN**: Simple but limited - fails on shifted targets
   - **NeuralSOC**: Most robust - guided sampling + control objective loss

3. **Three Components for Breaking Curse of Dimensionality**
   - Neural Networks (polynomial parameters in d)
   - Monte Carlo Integration (O(N^{-1/2}) regardless of d)
   - Smart Sampling (THE KEY!)

### References

1. Han, Jentzen, E (2018): "Solving high-dimensional PDEs using deep learning" (PNAS)
2. Raissi et al. (2018): "Forward-Backward Stochastic Neural Networks"
3. Ruthotto et al. (2020): "Machine learning framework for mean-field games" (PNAS)